In [1]:
pip install nltk

Note: you may need to restart the kernel to use updated packages.


In [2]:
import nltk
nltk.download('punkt')
nltk.download('punkt_tab')

[nltk_data] Downloading package punkt to /Users/msawant/nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package punkt_tab to
[nltk_data]     /Users/msawant/nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!


True

In [3]:
import pandas as pd
import numpy as np
from nltk.tokenize import word_tokenize
from sklearn.preprocessing import LabelEncoder
from sklearn.svm import SVC
from sklearn.model_selection import train_test_split
#from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn import metrics
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.model_selection import GridSearchCV
from sklearn.model_selection import RandomizedSearchCV
from sklearn.base import TransformerMixin, BaseEstimator
from sklearn.metrics import accuracy_score, classification_report,confusion_matrix,f1_score, precision_score, recall_score

In [4]:
pip install openpyxl

Note: you may need to restart the kernel to use updated packages.


In [5]:
import pandas as pd
#file_path = "EXIST_2023/disagreement_records_task1.xlsx"
#data = pd.read_csv("LLM_processed_dataset/All_LLM_EXIST2023.csv")




In [6]:
#len(data)

In [7]:
import re

def remove_urls_and_lower(text):
    # Define the regex pattern for URLs starting with http or https
    url_pattern = re.compile(r'http[s]?://\S+')
    # Substitute the URLs with an empty string
    cleaned_text = url_pattern.sub('', text)
    cleaned_text = cleaned_text.lower()
    return cleaned_text.strip()

In [8]:
def clean_text(text):
    return re.sub(r'[^A-Za-z\s]', '', text) if isinstance(text, str) else text
    #return re.sub(r'[^A-Za-z0-9\s]', '', text) if isinstance(text, str) else text

In [9]:
from nltk.corpus import stopwords
stop_words=set(stopwords.words('english'))
#additional_stopwords={'ago','also','along','always','amp','like','look','know','even','want','say','get','far','use','take','never','great','whole','know','even','use','take','would'}
stop_words.update(['ago','also','along','always','amp','like','look','know','even','want','say','get','far','use','take','never','great','whole','would','ok','dont','cant','shes','theyre','without','youre','isnt','us','yet','yall','u','id'])


#stop_words.update(['ago','also','along','always','amp','like','look','know','even','want','say','get','far','use','take','never','great','whole','would','ok','dont','cant','shes','theyre','without','youre'])

def stopwords_removal(text):
    words=word_tokenize(text)
    filtered_words=[w for w in words if w.lower() not in stop_words]
    return ' '.join(filtered_words)

In [10]:
from nltk.corpus import wordnet
def pos_tagger(nltk_tag):
    if nltk_tag.startswith('J'):
        return wordnet.ADJ
    elif nltk_tag.startswith('V'):
        return wordnet.VERB
    elif nltk_tag.startswith('N'):
        return wordnet.NOUN
    elif nltk_tag.startswith('R'):
        return wordnet.ADV
    else:          
        return None

In [11]:
from nltk.stem import WordNetLemmatizer
def apply_lemmatization(text):
    pos_tagged = nltk.pos_tag(nltk.word_tokenize(text))  
    wordnet_tagged = list(map(lambda x: (x[0], pos_tagger(x[1])), pos_tagged))
    lemmatizer = WordNetLemmatizer()
    lemmatized_sentence = []
    for word, tag in wordnet_tagged:
        if tag is None:
            # if there is no available tag, append the token as is
            lemmatized_sentence.append(word)
        else:        
        # else use the tag to lemmatize the token
            lemmatized_sentence.append(lemmatizer.lemmatize(word, tag))
    lemmatized_sentence = " ".join(lemmatized_sentence)
    return lemmatized_sentence

Function to execute logistic regression

In [12]:
def log_model(tweet,Y):
    tweet = tweet.reset_index(drop=True)
    Y = Y.reset_index(drop=True)
    
    tweet_processed=tweet.apply(remove_urls_and_lower)
    tweet_processed=tweet_processed.apply(clean_text)
    tweet_processed=tweet_processed.apply(apply_lemmatization)
    tweet_processed=tweet_processed.apply(stopwords_removal)
    
    vec=TfidfVectorizer(tokenizer=word_tokenize,token_pattern=None,ngram_range=(1,1))
    tweet_vectorized=vec.fit_transform(tweet_processed)
    
    X_train, X_test, y_train, y_test = train_test_split(tweet_vectorized,Y, test_size=0.20, random_state=42)
    
    logreg = LogisticRegression(verbose=1, random_state=10, penalty='l2', solver='newton-cg')
    #logreg = LogisticRegression()
    logreg.fit(X_train, y_train)
    
    y_pred_lg =logreg.predict(X_test)
    #y_pred_rf = y_pred_rf.astype(int)
    print("ACCURACY OF THE MODEL:", accuracy_score(y_test, y_pred_lg))
    print("F1 score:", f1_score(y_test, y_pred_lg, average='macro'))
    print(f"Precision: {precision_score(y_test, y_pred_lg, average='macro')}")
    print(f"Recall: {recall_score(y_test, y_pred_lg, average='macro')}")

    target_names = ['Yes', 'No']
    print("Classification Report for Human Annotators")
    print(classification_report(y_test, y_pred_lg, target_names=target_names))

**Age persona**

In [13]:
df1=pd.read_csv("LLM_annotation5_age_persona/GPT_5.4_EXIST2023_age18.csv")
print("GPT-5.4; Age 18")
log_model(df1["tweet"],df1["GPT_5.4"])

GPT-5.4; Age 18
Newton-CG iter = 0
  Check Convergence
    max |gradient| <= tol: 0.07438650306748468 <= 0.0001 False
Newton-CG iter = 1
  Check Convergence
    max |gradient| <= tol: 0.0017670790954557368 <= 0.0001 False
Newton-CG iter = 2
  Check Convergence
    max |gradient| <= tol: 0.0016467148706529682 <= 0.0001 False
Newton-CG iter = 3
  Check Convergence
    max |gradient| <= tol: 0.00022778400864675516 <= 0.0001 False
Newton-CG iter = 4
  Check Convergence
    max |gradient| <= tol: 1.6653028465478392e-06 <= 0.0001 True
  Solver did converge at loss = 0.5078612571987572.
ACCURACY OF THE MODEL: 0.7561349693251533
F1 score: 0.7321787025387583
Precision: 0.7879935555234935
Recall: 0.7293755446388386
Classification Report for Human Annotators
              precision    recall  f1-score   support

         Yes       0.72      0.93      0.81       369
          No       0.86      0.53      0.65       283

    accuracy                           0.76       652
   macro avg       0.79 

[Parallel(n_jobs=1)]: Using backend SequentialBackend with 1 concurrent workers.
[Parallel(n_jobs=1)]: Done   1 out of   1 | elapsed:    0.1s finished


In [14]:
df2=pd.read_csv("LLM_annotation5_age_persona/GPT_5.4_EXIST2023_age23.csv")
print("GPT-5.4; Age 23")
log_model(df2["tweet"],df2["GPT_5.4"])

GPT-5.4; Age 23
Newton-CG iter = 0
  Check Convergence
    max |gradient| <= tol: 0.07707055214723926 <= 0.0001 False
Newton-CG iter = 1
  Check Convergence
    max |gradient| <= tol: 0.0019249285087831049 <= 0.0001 False
Newton-CG iter = 2
  Check Convergence
    max |gradient| <= tol: 0.001776831359043522 <= 0.0001 False
Newton-CG iter = 3
  Check Convergence
    max |gradient| <= tol: 0.00023386043719042515 <= 0.0001 False
Newton-CG iter = 4
  Check Convergence
    max |gradient| <= tol: 1.716587947482931e-06 <= 0.0001 True
  Solver did converge at loss = 0.5091450350487413.
ACCURACY OF THE MODEL: 0.7576687116564417
F1 score: 0.7330500129567246
Precision: 0.7906492248062016
Recall: 0.729978915085298
Classification Report for Human Annotators
              precision    recall  f1-score   support

         Yes       0.72      0.94      0.81       370
          No       0.86      0.52      0.65       282

    accuracy                           0.76       652
   macro avg       0.79    

[Parallel(n_jobs=1)]: Using backend SequentialBackend with 1 concurrent workers.
[Parallel(n_jobs=1)]: Done   1 out of   1 | elapsed:    0.0s finished


In [15]:
df3=pd.read_csv("LLM_annotation5_age_persona/GPT_5.4_EXIST2023_age46.csv")
print("GPT-5.4; Age 46")
log_model(df3["tweet"],df3["GPT_5.4"])

GPT-5.4; Age 46
Newton-CG iter = 0
  Check Convergence
    max |gradient| <= tol: 0.07016871165644173 <= 0.0001 False
Newton-CG iter = 1
  Check Convergence
    max |gradient| <= tol: 0.0021865019922799774 <= 0.0001 False
Newton-CG iter = 2
  Check Convergence
    max |gradient| <= tol: 0.0017458840133784165 <= 0.0001 False
Newton-CG iter = 3
  Check Convergence
    max |gradient| <= tol: 0.0002486326242642674 <= 0.0001 False
Newton-CG iter = 4
  Check Convergence
    max |gradient| <= tol: 1.7495915105485496e-06 <= 0.0001 True
  Solver did converge at loss = 0.5129137347374451.
ACCURACY OF THE MODEL: 0.7638036809815951
F1 score: 0.7428785069704078
Precision: 0.7924012158054712
Recall: 0.739321800367422
Classification Report for Human Annotators
              precision    recall  f1-score   support

         Yes       0.73      0.93      0.82       368
          No       0.86      0.55      0.67       284

    accuracy                           0.76       652
   macro avg       0.79   

[Parallel(n_jobs=1)]: Using backend SequentialBackend with 1 concurrent workers.
[Parallel(n_jobs=1)]: Done   1 out of   1 | elapsed:    0.0s finished


In [16]:
df4=pd.read_csv("LLM_annotation5_age_persona/GPT_5.4_mini_EXIST2023_age18.csv")
print("GPT-5.4_mini; Age 18")
log_model(df4["tweet"],df4["GPT_5.4_mini"])

GPT-5.4_mini; Age 18
Newton-CG iter = 0
  Check Convergence
    max |gradient| <= tol: 0.011886503067484663 <= 0.0001 False
Newton-CG iter = 1
  Check Convergence
    max |gradient| <= tol: 0.008034301134005049 <= 0.0001 False
Newton-CG iter = 2
  Check Convergence
    max |gradient| <= tol: 0.0023657176761345666 <= 0.0001 False
Newton-CG iter = 3
  Check Convergence
    max |gradient| <= tol: 0.0003485423623771876 <= 0.0001 False
Newton-CG iter = 4
  Check Convergence
    max |gradient| <= tol: 2.117837391598168e-06 <= 0.0001 True
  Solver did converge at loss = 0.5282084847355948.
ACCURACY OF THE MODEL: 0.7607361963190185
F1 score: 0.7576138865852613
Precision: 0.7726675556872623
Recall: 0.7597560860850499
Classification Report for Human Annotators
              precision    recall  f1-score   support

         Yes       0.72      0.87      0.79       329
          No       0.83      0.65      0.73       323

    accuracy                           0.76       652
   macro avg       0.

[Parallel(n_jobs=1)]: Using backend SequentialBackend with 1 concurrent workers.
[Parallel(n_jobs=1)]: Done   1 out of   1 | elapsed:    0.0s finished


In [17]:
df5=pd.read_csv("LLM_annotation5_age_persona/GPT_5.4_mini_EXIST2023_age23.csv")
print("GPT-5.4_mini; Age 23")
log_model(df5["tweet"],df5["GPT_5.4_mini"])

GPT-5.4_mini; Age 23
Newton-CG iter = 0
  Check Convergence
    max |gradient| <= tol: 0.010352760736196322 <= 0.0001 False
Newton-CG iter = 1
  Check Convergence
    max |gradient| <= tol: 0.010485065349763222 <= 0.0001 False
Newton-CG iter = 2
  Check Convergence
    max |gradient| <= tol: 0.002454295663135632 <= 0.0001 False
Newton-CG iter = 3
  Check Convergence
    max |gradient| <= tol: 0.0003494750907386923 <= 0.0001 False
Newton-CG iter = 4
  Check Convergence
    max |gradient| <= tol: 2.06345657356808e-06 <= 0.0001 True
  Solver did converge at loss = 0.5291311833584132.
ACCURACY OF THE MODEL: 0.7822085889570553
F1 score: 0.7818617056367629
Precision: 0.7845713419027036
Recall: 0.7824826859379705
Classification Report for Human Annotators
              precision    recall  f1-score   support

         Yes       0.76      0.83      0.79       324
          No       0.81      0.74      0.77       328

    accuracy                           0.78       652
   macro avg       0.78

[Parallel(n_jobs=1)]: Using backend SequentialBackend with 1 concurrent workers.
[Parallel(n_jobs=1)]: Done   1 out of   1 | elapsed:    0.0s finished


In [18]:
df6=pd.read_csv("LLM_annotation5_age_persona/GPT_5.4_mini_EXIST2023_age46.csv")
print("GPT-5.4_mini; Age 46")
log_model(df6["tweet"],df6["GPT_5.4_mini"])

GPT-5.4_mini; Age 46
Newton-CG iter = 0
  Check Convergence
    max |gradient| <= tol: 0.018021472392638037 <= 0.0001 False
Newton-CG iter = 1
  Check Convergence
    max |gradient| <= tol: 0.011731818695408625 <= 0.0001 False
Newton-CG iter = 2
  Check Convergence
    max |gradient| <= tol: 0.002193444877781385 <= 0.0001 False
Newton-CG iter = 3
  Check Convergence
    max |gradient| <= tol: 0.0003121340493841603 <= 0.0001 False
Newton-CG iter = 4
  Check Convergence
    max |gradient| <= tol: 1.8689681803209929e-06 <= 0.0001 True
  Solver did converge at loss = 0.5270356554193034.
ACCURACY OF THE MODEL: 0.7837423312883436
F1 score: 0.7834728846493553
Precision: 0.7856574949598205
Recall: 0.7839882565492322
Classification Report for Human Annotators
              precision    recall  f1-score   support

         Yes       0.76      0.82      0.79       324
          No       0.81      0.74      0.78       328

    accuracy                           0.78       652
   macro avg       0.

[Parallel(n_jobs=1)]: Using backend SequentialBackend with 1 concurrent workers.
[Parallel(n_jobs=1)]: Done   1 out of   1 | elapsed:    0.0s finished


In [19]:
df7=pd.read_csv("LLM_annotation5_age_persona/GPT_5.4_nano_EXIST2023_age18.csv")
print("GPT-5.4_nano; Age 18")
log_model(df7["tweet"],df7["GPT_5.4_nano"])

GPT-5.4_nano; Age 18
Newton-CG iter = 0
  Check Convergence
    max |gradient| <= tol: 0.2691717791411043 <= 0.0001 False
Newton-CG iter = 1
  Check Convergence
    max |gradient| <= tol: 0.023069828041354476 <= 0.0001 False
Newton-CG iter = 2
  Check Convergence
    max |gradient| <= tol: 0.019365028891621717 <= 0.0001 False
Newton-CG iter = 3
  Check Convergence
    max |gradient| <= tol: 0.001267971986182544 <= 0.0001 False
Newton-CG iter = 4
  Check Convergence
    max |gradient| <= tol: 0.0004092320551153171 <= 0.0001 False
Newton-CG iter = 5
  Check Convergence
    max |gradient| <= tol: 3.40095033059658e-06 <= 0.0001 True
  Solver did converge at loss = 0.4294691392964035.
ACCURACY OF THE MODEL: 0.8144171779141104
F1 score: 0.5248128271384085
Precision: 0.8003806538289298
Recall: 0.5397675900810767
Classification Report for Human Annotators
              precision    recall  f1-score   support

         Yes       0.82      0.99      0.90       523
          No       0.79      0.

[Parallel(n_jobs=1)]: Using backend SequentialBackend with 1 concurrent workers.
[Parallel(n_jobs=1)]: Done   1 out of   1 | elapsed:    0.0s finished


In [20]:
df8=pd.read_csv("LLM_annotation5_age_persona/GPT_5.4_nano_EXIST2023_age23.csv")
print("GPT-5.4_nano; Age 23")
log_model(df8["tweet"],df8["GPT_5.4_nano"])

GPT-5.4_nano; Age 23
Newton-CG iter = 0
  Check Convergence
    max |gradient| <= tol: 0.24271472392638038 <= 0.0001 False
Newton-CG iter = 1
  Check Convergence
    max |gradient| <= tol: 0.030431234728277676 <= 0.0001 False
Newton-CG iter = 2
  Check Convergence
    max |gradient| <= tol: 0.001775123058680297 <= 0.0001 False
Newton-CG iter = 3
  Check Convergence
    max |gradient| <= tol: 0.000502862844426044 <= 0.0001 False
Newton-CG iter = 4
  Check Convergence
    max |gradient| <= tol: 2.6381902459574535e-06 <= 0.0001 True
  Solver did converge at loss = 0.4553260081200668.
ACCURACY OF THE MODEL: 0.7960122699386503
F1 score: 0.5236824036692208
Precision: 0.7253164556962026
Recall: 0.5399399778422875
Classification Report for Human Annotators
              precision    recall  f1-score   support

         Yes       0.80      0.99      0.88       513
          No       0.65      0.09      0.16       139

    accuracy                           0.80       652
   macro avg       0.73

[Parallel(n_jobs=1)]: Using backend SequentialBackend with 1 concurrent workers.
[Parallel(n_jobs=1)]: Done   1 out of   1 | elapsed:    0.0s finished


In [21]:
df9=pd.read_csv("LLM_annotation5_age_persona/GPT_5.4_nano_EXIST2023_age46.csv")
print("GPT-5.4_nano; Age 46")
log_model(df9["tweet"],df9["GPT_5.4_nano"])

GPT-5.4_nano; Age 46
Newton-CG iter = 0
  Check Convergence
    max |gradient| <= tol: 0.2825920245398773 <= 0.0001 False
Newton-CG iter = 1
  Check Convergence
    max |gradient| <= tol: 0.026503726435244932 <= 0.0001 False
Newton-CG iter = 2
  Check Convergence
    max |gradient| <= tol: 0.01888288484120348 <= 0.0001 False
Newton-CG iter = 3
  Check Convergence
    max |gradient| <= tol: 0.0011058485572019408 <= 0.0001 False
Newton-CG iter = 4
  Check Convergence
    max |gradient| <= tol: 0.0003389998825726048 <= 0.0001 False
Newton-CG iter = 5
  Check Convergence
    max |gradient| <= tol: 7.760109908966145e-06 <= 0.0001 True
  Solver did converge at loss = 0.4228834121851062.
ACCURACY OF THE MODEL: 0.8236196319018405
F1 score: 0.49839107834545315
Precision: 0.7872670807453417
Recall: 0.5233339114257335
Classification Report for Human Annotators
              precision    recall  f1-score   support

         Yes       0.82      1.00      0.90       533
          No       0.75      

[Parallel(n_jobs=1)]: Using backend SequentialBackend with 1 concurrent workers.
[Parallel(n_jobs=1)]: Done   1 out of   1 | elapsed:    0.0s finished


In [22]:
df10=pd.read_csv("LLM_annotation5_age_persona/mistral_EXIST2023_age18.csv")
print("mistral; Age 18")
log_model(df10["tweet"],df10["mistral"])

mistral; Age 18
Newton-CG iter = 0
  Check Convergence
    max |gradient| <= tol: 0.138420245398773 <= 0.0001 False
Newton-CG iter = 1
  Check Convergence
    max |gradient| <= tol: 0.02028115946564031 <= 0.0001 False
Newton-CG iter = 2
  Check Convergence
    max |gradient| <= tol: 0.00310321761310194 <= 0.0001 False
Newton-CG iter = 3
  Check Convergence
    max |gradient| <= tol: 0.0003686800635746782 <= 0.0001 False
Newton-CG iter = 4
  Check Convergence
    max |gradient| <= tol: 1.6707702801994906e-06 <= 0.0001 True
  Solver did converge at loss = 0.514549724791273.
ACCURACY OF THE MODEL: 0.691717791411043
F1 score: 0.5930858248230936
Precision: 0.7307578292078684
Recall: 0.608771159374002
Classification Report for Human Annotators
              precision    recall  f1-score   support

         Yes       0.78      0.26      0.39       248
          No       0.68      0.96      0.79       404

    accuracy                           0.69       652
   macro avg       0.73      0.61 

[Parallel(n_jobs=1)]: Using backend SequentialBackend with 1 concurrent workers.
[Parallel(n_jobs=1)]: Done   1 out of   1 | elapsed:    0.0s finished


In [23]:
df11=pd.read_csv("LLM_annotation5_age_persona/mistral_EXIST2023_age23.csv")
print("mistral; Age 23")
log_model(df11["tweet"],df11["mistral"])

mistral; Age 23
Newton-CG iter = 0
  Check Convergence
    max |gradient| <= tol: 0.15950920245398775 <= 0.0001 False
Newton-CG iter = 1
  Check Convergence
    max |gradient| <= tol: 0.02255039752926424 <= 0.0001 False
Newton-CG iter = 2
  Check Convergence
    max |gradient| <= tol: 0.0031160951312991635 <= 0.0001 False
Newton-CG iter = 3
  Check Convergence
    max |gradient| <= tol: 0.00032738438607924194 <= 0.0001 False
Newton-CG iter = 4
  Check Convergence
    max |gradient| <= tol: 1.431134762798665e-06 <= 0.0001 True
  Solver did converge at loss = 0.5054339121919036.
ACCURACY OF THE MODEL: 0.6932515337423313
F1 score: 0.5498481082573875
Precision: 0.760531561461794
Recall: 0.5801742117531591
Classification Report for Human Annotators
              precision    recall  f1-score   support

         Yes       0.84      0.18      0.30       234
          No       0.68      0.98      0.80       418

    accuracy                           0.69       652
   macro avg       0.76     

[Parallel(n_jobs=1)]: Using backend SequentialBackend with 1 concurrent workers.
[Parallel(n_jobs=1)]: Done   1 out of   1 | elapsed:    0.0s finished


In [24]:
df12=pd.read_csv("LLM_annotation5_age_persona/mistral_EXIST2023_age46.csv")
print("mistral; Age 46")
log_model(df12["tweet"],df12["mistral"])

mistral; Age 46
Newton-CG iter = 0
  Check Convergence
    max |gradient| <= tol: 0.1441717791411043 <= 0.0001 False
Newton-CG iter = 1
  Check Convergence
    max |gradient| <= tol: 0.0208345832257466 <= 0.0001 False
Newton-CG iter = 2
  Check Convergence
    max |gradient| <= tol: 0.0031455756273057987 <= 0.0001 False
Newton-CG iter = 3
  Check Convergence
    max |gradient| <= tol: 0.00036779125434917726 <= 0.0001 False
Newton-CG iter = 4
  Check Convergence
    max |gradient| <= tol: 1.654281155553071e-06 <= 0.0001 True
  Solver did converge at loss = 0.5123136047298988.
ACCURACY OF THE MODEL: 0.6993865030674846
F1 score: 0.5889081901820755
Precision: 0.7680165063265966
Recall: 0.6081231509802938
Classification Report for Human Annotators
              precision    recall  f1-score   support

         Yes       0.86      0.24      0.38       245
          No       0.68      0.98      0.80       407

    accuracy                           0.70       652
   macro avg       0.77      

[Parallel(n_jobs=1)]: Using backend SequentialBackend with 1 concurrent workers.
[Parallel(n_jobs=1)]: Done   1 out of   1 | elapsed:    0.0s finished


In [25]:
df13=pd.read_csv("LLM_annotation5_age_persona/phi_EXIST2023_age18.csv")
print("phi; Age 18")
log_model(df13["tweet"],df13["phi"])

phi; Age 18
Newton-CG iter = 0
  Check Convergence
    max |gradient| <= tol: 0.15874233128834356 <= 0.0001 False
Newton-CG iter = 1
  Check Convergence
    max |gradient| <= tol: 0.013637119938742635 <= 0.0001 False
Newton-CG iter = 2
  Check Convergence
    max |gradient| <= tol: 0.002267997973770434 <= 0.0001 False
Newton-CG iter = 3
  Check Convergence
    max |gradient| <= tol: 0.00013564930589873587 <= 0.0001 False
Newton-CG iter = 4
  Check Convergence
    max |gradient| <= tol: 2.639058174085801e-06 <= 0.0001 True
  Solver did converge at loss = 0.507736350927853.
ACCURACY OF THE MODEL: 0.7208588957055214
F1 score: 0.5882638686485961
Precision: 0.7272360999194198
Recall: 0.5962744847010626
Classification Report for Human Annotators
              precision    recall  f1-score   support

         Yes       0.72      0.96      0.82       438
          No       0.74      0.23      0.35       214

    accuracy                           0.72       652
   macro avg       0.73      0.6

[Parallel(n_jobs=1)]: Using backend SequentialBackend with 1 concurrent workers.
[Parallel(n_jobs=1)]: Done   1 out of   1 | elapsed:    0.0s finished


In [26]:
df14=pd.read_csv("LLM_annotation5_age_persona/phi_EXIST2023_age23.csv")
print("phi; Age 23")
log_model(df14["tweet"],df14["phi"])

phi; Age 23
Newton-CG iter = 0
  Check Convergence
    max |gradient| <= tol: 0.15069018404907975 <= 0.0001 False
Newton-CG iter = 1
  Check Convergence
    max |gradient| <= tol: 0.012109244958699077 <= 0.0001 False
Newton-CG iter = 2
  Check Convergence
    max |gradient| <= tol: 0.002283654507258608 <= 0.0001 False
Newton-CG iter = 3
  Check Convergence
    max |gradient| <= tol: 9.016928967750353e-05 <= 0.0001 True
  Solver did converge at loss = 0.5124339220557632.
ACCURACY OF THE MODEL: 0.7177914110429447
F1 score: 0.599823875538714
Precision: 0.7317708333333333
Recall: 0.6062853551225644
Classification Report for Human Annotators
              precision    recall  f1-score   support

         Yes       0.71      0.96      0.82       430
          No       0.75      0.26      0.38       222

    accuracy                           0.72       652
   macro avg       0.73      0.61      0.60       652
weighted avg       0.73      0.72      0.67       652



[Parallel(n_jobs=1)]: Using backend SequentialBackend with 1 concurrent workers.
[Parallel(n_jobs=1)]: Done   1 out of   1 | elapsed:    0.0s finished


In [27]:
df15=pd.read_csv("LLM_annotation5_age_persona/phi_EXIST2023_age46.csv")
print("phi; Age 46")
log_model(df15["tweet"],df15["phi"])

phi; Age 46
Newton-CG iter = 0
  Check Convergence
    max |gradient| <= tol: 0.17331288343558282 <= 0.0001 False
Newton-CG iter = 1
  Check Convergence
    max |gradient| <= tol: 0.016288654475296177 <= 0.0001 False
Newton-CG iter = 2
  Check Convergence
    max |gradient| <= tol: 0.0022279363080609195 <= 0.0001 False
Newton-CG iter = 3
  Check Convergence
    max |gradient| <= tol: 0.00020617478272599984 <= 0.0001 False
Newton-CG iter = 4
  Check Convergence
    max |gradient| <= tol: 2.521175009079235e-06 <= 0.0001 True
  Solver did converge at loss = 0.5003301933797536.
ACCURACY OF THE MODEL: 0.7239263803680982
F1 score: 0.5796260477111541
Precision: 0.7146577380952381
Recall: 0.5879228525403805
Classification Report for Human Annotators
              precision    recall  f1-score   support

         Yes       0.73      0.96      0.83       446
          No       0.70      0.22      0.33       206

    accuracy                           0.72       652
   macro avg       0.71      0

[Parallel(n_jobs=1)]: Using backend SequentialBackend with 1 concurrent workers.
[Parallel(n_jobs=1)]: Done   1 out of   1 | elapsed:    0.0s finished


In [28]:
df16=pd.read_csv("LLM_annotation5_age_persona/GPT_5.5_EXIST2023_age18.csv")
print("GPT 5.5; Age 18")
log_model(df16["tweet"],df16["GPT_5.5"])

GPT 5.5; Age 18
Newton-CG iter = 0
  Check Convergence
    max |gradient| <= tol: 0.0272239263803681 <= 0.0001 False
Newton-CG iter = 1
  Check Convergence
    max |gradient| <= tol: 0.005501678614446411 <= 0.0001 False
Newton-CG iter = 2
  Check Convergence
    max |gradient| <= tol: 0.003045857646399899 <= 0.0001 False
Newton-CG iter = 3
  Check Convergence
    max |gradient| <= tol: 0.00033619104927855824 <= 0.0001 False
Newton-CG iter = 4
  Check Convergence
    max |gradient| <= tol: 1.9092157440732394e-06 <= 0.0001 True
  Solver did converge at loss = 0.5217197898164995.
ACCURACY OF THE MODEL: 0.7883435582822086
F1 score: 0.7806640339330115
Precision: 0.7966346398251233
Recall: 0.7773679256942397
Classification Report for Human Annotators
              precision    recall  f1-score   support

         Yes       0.77      0.89      0.82       359
          No       0.83      0.67      0.74       293

    accuracy                           0.79       652
   macro avg       0.80    

[Parallel(n_jobs=1)]: Using backend SequentialBackend with 1 concurrent workers.
[Parallel(n_jobs=1)]: Done   1 out of   1 | elapsed:    0.0s finished


In [29]:
df17=pd.read_csv("LLM_annotation5_age_persona/GPT_5.5_EXIST2023_age23.csv")
print("GPT_5.5 ; Age 23")
log_model(df17["tweet"],df17["GPT_5.5"])

GPT_5.5 ; Age 23
Newton-CG iter = 0
  Check Convergence
    max |gradient| <= tol: 0.021088957055214727 <= 0.0001 False
Newton-CG iter = 1
  Check Convergence
    max |gradient| <= tol: 0.006378450086687634 <= 0.0001 False
Newton-CG iter = 2
  Check Convergence
    max |gradient| <= tol: 0.0029688658868676585 <= 0.0001 False
Newton-CG iter = 3
  Check Convergence
    max |gradient| <= tol: 0.0003481834061613365 <= 0.0001 False
Newton-CG iter = 4
  Check Convergence
    max |gradient| <= tol: 1.994573303511389e-06 <= 0.0001 True
  Solver did converge at loss = 0.5227279967710425.
ACCURACY OF THE MODEL: 0.7837423312883436
F1 score: 0.7775244843583247
Precision: 0.7910394624680339
Recall: 0.7751014294922838
Classification Report for Human Annotators
              precision    recall  f1-score   support

         Yes       0.76      0.88      0.81       354
          No       0.82      0.67      0.74       298

    accuracy                           0.78       652
   macro avg       0.79  

[Parallel(n_jobs=1)]: Using backend SequentialBackend with 1 concurrent workers.
[Parallel(n_jobs=1)]: Done   1 out of   1 | elapsed:    0.0s finished


In [30]:
df18=pd.read_csv("LLM_annotation5_age_persona/GPT_5.5_EXIST2023_age46.csv")
print("GPT_5.5; Age 46")
log_model(df18["tweet"],df18["GPT_5.5"])

GPT_5.5; Age 46
Newton-CG iter = 0
  Check Convergence
    max |gradient| <= tol: 0.02569018404907976 <= 0.0001 False
Newton-CG iter = 1
  Check Convergence
    max |gradient| <= tol: 0.005710756567510307 <= 0.0001 False
Newton-CG iter = 2
  Check Convergence
    max |gradient| <= tol: 0.002965438035155642 <= 0.0001 False
Newton-CG iter = 3
  Check Convergence
    max |gradient| <= tol: 0.00034485836135114516 <= 0.0001 False
Newton-CG iter = 4
  Check Convergence
    max |gradient| <= tol: 2.001859760306958e-06 <= 0.0001 True
  Solver did converge at loss = 0.5247438854518935.
ACCURACY OF THE MODEL: 0.7760736196319018
F1 score: 0.7687538868159205
Precision: 0.785267730280518
Recall: 0.766446744776855
Classification Report for Human Annotators
              precision    recall  f1-score   support

         Yes       0.75      0.88      0.81       354
          No       0.82      0.65      0.73       298

    accuracy                           0.78       652
   macro avg       0.79      

[Parallel(n_jobs=1)]: Using backend SequentialBackend with 1 concurrent workers.
[Parallel(n_jobs=1)]: Done   1 out of   1 | elapsed:    0.0s finished


**PROMPT 4 Male Persona**

In [31]:
df19=pd.read_csv("LLM_annotation4_male_persona/GPT_5.5_EXIST2023.csv")
print("GPT-5.5")
log_model(df19["tweet"],df19["GPT_5.5"])

GPT-5.5
Newton-CG iter = 0
  Check Convergence
    max |gradient| <= tol: 0.03144171779141105 <= 0.0001 False
Newton-CG iter = 1
  Check Convergence
    max |gradient| <= tol: 0.005135466699433231 <= 0.0001 False
Newton-CG iter = 2
  Check Convergence
    max |gradient| <= tol: 0.0028904409303069454 <= 0.0001 False
Newton-CG iter = 3
  Check Convergence
    max |gradient| <= tol: 0.0003286670472404012 <= 0.0001 False
Newton-CG iter = 4
  Check Convergence
    max |gradient| <= tol: 1.9240723151984704e-06 <= 0.0001 True
  Solver did converge at loss = 0.5226032267134476.
ACCURACY OF THE MODEL: 0.7791411042944786
F1 score: 0.769701726844584
Precision: 0.7915160579999169
Recall: 0.7666552654581386
Classification Report for Human Annotators
              precision    recall  f1-score   support

         Yes       0.75      0.89      0.82       358
          No       0.83      0.64      0.72       294

    accuracy                           0.78       652
   macro avg       0.79      0.77  

[Parallel(n_jobs=1)]: Using backend SequentialBackend with 1 concurrent workers.
[Parallel(n_jobs=1)]: Done   1 out of   1 | elapsed:    0.0s finished


In [32]:
df20=pd.read_csv("LLM_annotation4_male_persona/GPT_5.4_EXIST2023.csv")
print("GPT-5.4")
log_model(df20["tweet"],df20["GPT_5.4"])

GPT-5.4
Newton-CG iter = 0
  Check Convergence
    max |gradient| <= tol: 0.0763036809815951 <= 0.0001 False
Newton-CG iter = 1
  Check Convergence
    max |gradient| <= tol: 0.0019252910034659322 <= 0.0001 False
Newton-CG iter = 2
  Check Convergence
    max |gradient| <= tol: 0.0017750495937415996 <= 0.0001 False
Newton-CG iter = 3
  Check Convergence
    max |gradient| <= tol: 0.0002520011676867195 <= 0.0001 False
Newton-CG iter = 4
  Check Convergence
    max |gradient| <= tol: 1.771098806736468e-06 <= 0.0001 True
  Solver did converge at loss = 0.5104291145467154.
ACCURACY OF THE MODEL: 0.7484662576687117
F1 score: 0.720572402186752
Precision: 0.7763730138312078
Recall: 0.717179302045728
Classification Report for Human Annotators
              precision    recall  f1-score   support

         Yes       0.72      0.93      0.81       375
          No       0.83      0.51      0.63       277

    accuracy                           0.75       652
   macro avg       0.78      0.72    

[Parallel(n_jobs=1)]: Using backend SequentialBackend with 1 concurrent workers.
[Parallel(n_jobs=1)]: Done   1 out of   1 | elapsed:    0.0s finished


In [33]:
df21=pd.read_csv("LLM_annotation4_male_persona/GPT_5.4_mini_EXIST2023.csv")
print("GPT-5.4-mini")
log_model(df21["tweet"],df21["GPT_5.4_mini"])

GPT-5.4-mini
Newton-CG iter = 0
  Check Convergence
    max |gradient| <= tol: 0.010110656731531937 <= 0.0001 False
Newton-CG iter = 1
  Check Convergence
    max |gradient| <= tol: 0.010352794661394379 <= 0.0001 False
Newton-CG iter = 2
  Check Convergence
    max |gradient| <= tol: 0.0021062899653690875 <= 0.0001 False
Newton-CG iter = 3
  Check Convergence
    max |gradient| <= tol: 0.00031713176235679055 <= 0.0001 False
Newton-CG iter = 4
  Check Convergence
    max |gradient| <= tol: 1.913057230606627e-06 <= 0.0001 True
  Solver did converge at loss = 0.5248758191312279.
ACCURACY OF THE MODEL: 0.799079754601227
F1 score: 0.7983582910551179
Precision: 0.8072347622204925
Recall: 0.8010300154408165
Classification Report for Human Annotators
              precision    recall  f1-score   support

         Yes       0.75      0.88      0.81       318
          No       0.86      0.72      0.79       334

    accuracy                           0.80       652
   macro avg       0.81      

[Parallel(n_jobs=1)]: Using backend SequentialBackend with 1 concurrent workers.
[Parallel(n_jobs=1)]: Done   1 out of   1 | elapsed:    0.0s finished


In [34]:
df22=pd.read_csv("LLM_annotation4_male_persona/GPT_5.4_nano_EXIST2023.csv")
print("GPT-5.4-nano")
log_model(df22["tweet"],df22["GPT_5.4_nano"])

GPT-5.4-nano
Newton-CG iter = 0
  Check Convergence
    max |gradient| <= tol: 0.263420245398773 <= 0.0001 False
Newton-CG iter = 1
  Check Convergence
    max |gradient| <= tol: 0.021698825090368357 <= 0.0001 False
Newton-CG iter = 2
  Check Convergence
    max |gradient| <= tol: 0.018067401381948657 <= 0.0001 False
Newton-CG iter = 3
  Check Convergence
    max |gradient| <= tol: 0.0011138328733935614 <= 0.0001 False
Newton-CG iter = 4
  Check Convergence
    max |gradient| <= tol: 0.00035500343672899607 <= 0.0001 False
Newton-CG iter = 5
  Check Convergence
    max |gradient| <= tol: 8.834509496431038e-06 <= 0.0001 True
  Solver did converge at loss = 0.4414157800151931.
ACCURACY OF THE MODEL: 0.7929447852760736
F1 score: 0.4824757906619865
Precision: 0.730689476412649
Recall: 0.5188208424970394
Classification Report for Human Annotators
              precision    recall  f1-score   support

         Yes       0.79      0.99      0.88       514
          No       0.67      0.04     

[Parallel(n_jobs=1)]: Using backend SequentialBackend with 1 concurrent workers.
[Parallel(n_jobs=1)]: Done   1 out of   1 | elapsed:    0.0s finished


In [35]:
df23=pd.read_csv("LLM_annotation4_male_persona/mistral_EXIST2023.csv")
print("Mistral")
log_model(df23["tweet"],df23["mistral"])

Mistral
Newton-CG iter = 0
  Check Convergence
    max |gradient| <= tol: 0.11042944785276074 <= 0.0001 False
Newton-CG iter = 1
  Check Convergence
    max |gradient| <= tol: 0.017214988576281485 <= 0.0001 False
Newton-CG iter = 2
  Check Convergence
    max |gradient| <= tol: 0.0032053387673205216 <= 0.0001 False
Newton-CG iter = 3
  Check Convergence
    max |gradient| <= tol: 0.0004264089428699454 <= 0.0001 False
Newton-CG iter = 4
  Check Convergence
    max |gradient| <= tol: 2.048437079622649e-06 <= 0.0001 True
  Solver did converge at loss = 0.5251785324742304.
ACCURACY OF THE MODEL: 0.7223926380368099
F1 score: 0.6749902920108068
Precision: 0.7569254445964433
Recall: 0.6753976360717934
Classification Report for Human Annotators
              precision    recall  f1-score   support

         Yes       0.82      0.42      0.55       267
          No       0.70      0.94      0.80       385

    accuracy                           0.72       652
   macro avg       0.76      0.68  

[Parallel(n_jobs=1)]: Using backend SequentialBackend with 1 concurrent workers.
[Parallel(n_jobs=1)]: Done   1 out of   1 | elapsed:    0.0s finished


In [36]:
df24=pd.read_csv("LLM_annotation4_male_persona/phi_EXIST2023.csv")
print("Phi")
log_model(df24["tweet"],df24["phi"])

Phi
Newton-CG iter = 0
  Check Convergence
    max |gradient| <= tol: 0.2039877300613497 <= 0.0001 False
Newton-CG iter = 1
  Check Convergence
    max |gradient| <= tol: 0.021682743908292564 <= 0.0001 False
Newton-CG iter = 2
  Check Convergence
    max |gradient| <= tol: 0.0021825205906901256 <= 0.0001 False
Newton-CG iter = 3
  Check Convergence
    max |gradient| <= tol: 0.0003101723619348333 <= 0.0001 False
Newton-CG iter = 4
  Check Convergence
    max |gradient| <= tol: 1.715660235529937e-06 <= 0.0001 True
  Solver did converge at loss = 0.4841321108053557.
ACCURACY OF THE MODEL: 0.7361963190184049
F1 score: 0.536441798941799
Precision: 0.7118121035487742
Recall: 0.5570169366375929
Classification Report for Human Annotators
              precision    recall  f1-score   support

         Yes       0.74      0.97      0.84       466
          No       0.68      0.14      0.23       186

    accuracy                           0.74       652
   macro avg       0.71      0.56      0.

[Parallel(n_jobs=1)]: Using backend SequentialBackend with 1 concurrent workers.
[Parallel(n_jobs=1)]: Done   1 out of   1 | elapsed:    0.0s finished


**PROMPT 3 Female PERSONA**

In [37]:
df25=pd.read_csv("LLM_annotation3_female_persona/GPT_5.5_EXIST2023.csv")
print("GPT-5.5")
log_model(df25["tweet"],df25["GPT_5.5"])

GPT-5.5
Newton-CG iter = 0
  Check Convergence
    max |gradient| <= tol: 0.02070552147239264 <= 0.0001 False
Newton-CG iter = 1
  Check Convergence
    max |gradient| <= tol: 0.006400697667707535 <= 0.0001 False
Newton-CG iter = 2
  Check Convergence
    max |gradient| <= tol: 0.0030006148919843752 <= 0.0001 False
Newton-CG iter = 3
  Check Convergence
    max |gradient| <= tol: 0.00034638711992789496 <= 0.0001 False
Newton-CG iter = 4
  Check Convergence
    max |gradient| <= tol: 1.9632345008915864e-06 <= 0.0001 True
  Solver did converge at loss = 0.520986760934665.
ACCURACY OF THE MODEL: 0.7822085889570553
F1 score: 0.7755865813457437
Precision: 0.7861106675183647
Recall: 0.7726978691019787
Classification Report for Human Annotators
              precision    recall  f1-score   support

         Yes       0.77      0.86      0.81       360
          No       0.80      0.68      0.74       292

    accuracy                           0.78       652
   macro avg       0.79      0.77 

[Parallel(n_jobs=1)]: Using backend SequentialBackend with 1 concurrent workers.
[Parallel(n_jobs=1)]: Done   1 out of   1 | elapsed:    0.0s finished


In [38]:
df26=pd.read_csv("LLM_annotation3_female_persona/GPT_5.4_EXIST2023.csv")
print("GPT-5.4")
log_model(df26["tweet"],df26["GPT_5.4"])

GPT-5.4
Newton-CG iter = 0
  Check Convergence
    max |gradient| <= tol: 0.06365030674846626 <= 0.0001 False
Newton-CG iter = 1
  Check Convergence
    max |gradient| <= tol: 0.002841696851224926 <= 0.0001 False
Newton-CG iter = 2
  Check Convergence
    max |gradient| <= tol: 0.001806567605074195 <= 0.0001 False
Newton-CG iter = 3
  Check Convergence
    max |gradient| <= tol: 0.00027555871362664776 <= 0.0001 False
Newton-CG iter = 4
  Check Convergence
    max |gradient| <= tol: 1.9430770818174015e-06 <= 0.0001 True
  Solver did converge at loss = 0.5153352315207508.
ACCURACY OF THE MODEL: 0.7730061349693251
F1 score: 0.7524220032840723
Precision: 0.790248348143085
Recall: 0.7459931255551693
Classification Report for Human Annotators
              precision    recall  f1-score   support

         Yes       0.75      0.92      0.82       378
          No       0.83      0.58      0.68       274

    accuracy                           0.77       652
   macro avg       0.79      0.75  

[Parallel(n_jobs=1)]: Using backend SequentialBackend with 1 concurrent workers.
[Parallel(n_jobs=1)]: Done   1 out of   1 | elapsed:    0.0s finished


In [39]:
df27=pd.read_csv("LLM_annotation3_female_persona/GPT_5.4_mini_EXIST2023.csv")
print("GPT-5.4-mini")
log_model(df27["tweet"],df27["GPT_5.4_mini"])

GPT-5.4-mini
Newton-CG iter = 0
  Check Convergence
    max |gradient| <= tol: 0.009906414847833873 <= 0.0001 False
Newton-CG iter = 1
  Check Convergence
    max |gradient| <= tol: 0.010019005442152357 <= 0.0001 False
Newton-CG iter = 2
  Check Convergence
    max |gradient| <= tol: 0.0021611744395759556 <= 0.0001 False
Newton-CG iter = 3
  Check Convergence
    max |gradient| <= tol: 0.0003307016886443799 <= 0.0001 False
Newton-CG iter = 4
  Check Convergence
    max |gradient| <= tol: 2.042058240392386e-06 <= 0.0001 True
  Solver did converge at loss = 0.5297539901384979.
ACCURACY OF THE MODEL: 0.7837423312883436
F1 score: 0.7812928965154647
Precision: 0.78884041109172
Recall: 0.7807637573392896
Classification Report for Human Annotators
              precision    recall  f1-score   support

         Yes       0.76      0.86      0.80       339
          No       0.82      0.71      0.76       313

    accuracy                           0.78       652
   macro avg       0.79      0.

[Parallel(n_jobs=1)]: Using backend SequentialBackend with 1 concurrent workers.
[Parallel(n_jobs=1)]: Done   1 out of   1 | elapsed:    0.0s finished


In [40]:
df28=pd.read_csv("LLM_annotation3_female_persona/GPT_5.4_nano_EXIST2023.csv")
print("GPT-5.4-nano")
log_model(df28["tweet"],df28["GPT_5.4_nano"])

GPT-5.4-nano
Newton-CG iter = 0
  Check Convergence
    max |gradient| <= tol: 0.2657208588957055 <= 0.0001 False
Newton-CG iter = 1
  Check Convergence
    max |gradient| <= tol: 0.022246550096018225 <= 0.0001 False
Newton-CG iter = 2
  Check Convergence
    max |gradient| <= tol: 0.01890351656231779 <= 0.0001 False
Newton-CG iter = 3
  Check Convergence
    max |gradient| <= tol: 0.0011804079488558052 <= 0.0001 False
Newton-CG iter = 4
  Check Convergence
    max |gradient| <= tol: 0.0003681002190508414 <= 0.0001 False
Newton-CG iter = 5
  Check Convergence
    max |gradient| <= tol: 2.9302659694501864e-06 <= 0.0001 True
  Solver did converge at loss = 0.43536298035920123.
ACCURACY OF THE MODEL: 0.8236196319018405
F1 score: 0.5053273956787069
Precision: 0.6839412543637895
Recall: 0.5245753988677303
Classification Report for Human Annotators
              precision    recall  f1-score   support

         Yes       0.83      0.99      0.90       536
          No       0.54      0.06   

[Parallel(n_jobs=1)]: Using backend SequentialBackend with 1 concurrent workers.
[Parallel(n_jobs=1)]: Done   1 out of   1 | elapsed:    0.0s finished


In [41]:
df29=pd.read_csv("LLM_annotation3_female_persona/mistral_EXIST2023.csv")
print("Mistral")
log_model(df29["tweet"],df29["mistral"])

Mistral
Newton-CG iter = 0
  Check Convergence
    max |gradient| <= tol: 0.1334355828220859 <= 0.0001 False
Newton-CG iter = 1
  Check Convergence
    max |gradient| <= tol: 0.01931704278455911 <= 0.0001 False
Newton-CG iter = 2
  Check Convergence
    max |gradient| <= tol: 0.003205792269454633 <= 0.0001 False
Newton-CG iter = 3
  Check Convergence
    max |gradient| <= tol: 0.000404848884003905 <= 0.0001 False
Newton-CG iter = 4
  Check Convergence
    max |gradient| <= tol: 1.8689421265845677e-06 <= 0.0001 True
  Solver did converge at loss = 0.5189540753520602.
ACCURACY OF THE MODEL: 0.700920245398773
F1 score: 0.6067952607602454
Precision: 0.7610825845100556
Recall: 0.6212407228939603
Classification Report for Human Annotators
              precision    recall  f1-score   support

         Yes       0.84      0.27      0.41       251
          No       0.68      0.97      0.80       401

    accuracy                           0.70       652
   macro avg       0.76      0.62      

[Parallel(n_jobs=1)]: Using backend SequentialBackend with 1 concurrent workers.
[Parallel(n_jobs=1)]: Done   1 out of   1 | elapsed:    0.0s finished


In [42]:
df30=pd.read_csv("LLM_annotation3_female_persona/phi_EXIST2023.csv")
print("Phi")
log_model(df30["tweet"],df30["phi"])

Phi
Newton-CG iter = 0
  Check Convergence
    max |gradient| <= tol: 0.2043711656441718 <= 0.0001 False
Newton-CG iter = 1
  Check Convergence
    max |gradient| <= tol: 0.021731291485319215 <= 0.0001 False
Newton-CG iter = 2
  Check Convergence
    max |gradient| <= tol: 0.002177305862148945 <= 0.0001 False
Newton-CG iter = 3
  Check Convergence
    max |gradient| <= tol: 0.00031260300123542306 <= 0.0001 False
Newton-CG iter = 4
  Check Convergence
    max |gradient| <= tol: 1.717402333609717e-06 <= 0.0001 True
  Solver did converge at loss = 0.48434066590177144.
ACCURACY OF THE MODEL: 0.7377300613496932
F1 score: 0.5374984962437206
Precision: 0.7044589450788472
Recall: 0.5567632850241546
Classification Report for Human Annotators
              precision    recall  f1-score   support

         Yes       0.74      0.97      0.84       468
          No       0.67      0.14      0.23       184

    accuracy                           0.74       652
   macro avg       0.70      0.56      

[Parallel(n_jobs=1)]: Using backend SequentialBackend with 1 concurrent workers.
[Parallel(n_jobs=1)]: Done   1 out of   1 | elapsed:    0.0s finished


**PROMPT2**

In [43]:
#PROMPT2
print("GPT-5.4")
df31 = pd.read_csv("LLM_annotation2/GPT_5.4_EXIST2023.csv")
log_model(df31["tweet"],df31["GPT_5.4"])

GPT-5.4
Newton-CG iter = 0
  Check Convergence
    max |gradient| <= tol: 0.08934049079754602 <= 0.0001 False
Newton-CG iter = 1
  Check Convergence
    max |gradient| <= tol: 0.0020690663352081508 <= 0.0001 False
Newton-CG iter = 2
  Check Convergence
    max |gradient| <= tol: 0.0018931220513278123 <= 0.0001 False
Newton-CG iter = 3
  Check Convergence
    max |gradient| <= tol: 0.00023378618485568733 <= 0.0001 False
Newton-CG iter = 4
  Check Convergence
    max |gradient| <= tol: 1.5985467362602222e-06 <= 0.0001 True
  Solver did converge at loss = 0.5110305180445208.
ACCURACY OF THE MODEL: 0.7331288343558282
F1 score: 0.6936718977826262
Precision: 0.7632727127169867
Recall: 0.6915177574810487
Classification Report for Human Annotators
              precision    recall  f1-score   support

         Yes       0.71      0.93      0.80       383
          No       0.82      0.45      0.58       269

    accuracy                           0.73       652
   macro avg       0.76      0.6

[Parallel(n_jobs=1)]: Using backend SequentialBackend with 1 concurrent workers.
[Parallel(n_jobs=1)]: Done   1 out of   1 | elapsed:    0.0s finished


In [44]:
print("GPT-5.4-mini")
df32 = pd.read_csv("LLM_annotation2/GPT_5.4_mini_EXIST2023.csv")
log_model(df32["tweet"],df32["GPT_5.4_mini"])

GPT-5.4-mini
Newton-CG iter = 0
  Check Convergence
    max |gradient| <= tol: 0.01045816867856806 <= 0.0001 False
Newton-CG iter = 1
  Check Convergence
    max |gradient| <= tol: 0.011029268698000815 <= 0.0001 False
Newton-CG iter = 2
  Check Convergence
    max |gradient| <= tol: 0.0016766076208495724 <= 0.0001 False
Newton-CG iter = 3
  Check Convergence
    max |gradient| <= tol: 0.0002542472585080214 <= 0.0001 False
Newton-CG iter = 4
  Check Convergence
    max |gradient| <= tol: 1.6322803600794453e-06 <= 0.0001 True
  Solver did converge at loss = 0.5215386419701735.
ACCURACY OF THE MODEL: 0.7837423312883436
F1 score: 0.7811471136472387
Precision: 0.7904768397678132
Recall: 0.7809570910461118
Classification Report for Human Annotators
              precision    recall  f1-score   support

         Yes       0.75      0.86      0.80       337
          No       0.83      0.70      0.76       315

    accuracy                           0.78       652
   macro avg       0.79      

[Parallel(n_jobs=1)]: Using backend SequentialBackend with 1 concurrent workers.
[Parallel(n_jobs=1)]: Done   1 out of   1 | elapsed:    0.0s finished


In [45]:
print("GPT-5.4-nano")
df33 = pd.read_csv("LLM_annotation2/GPT_5.4_nano_EXIST2023.csv")
log_model(df33["tweet"],df33["GPT_5.4_nano"])

GPT-5.4-nano
Newton-CG iter = 0
  Check Convergence
    max |gradient| <= tol: 0.23121165644171782 <= 0.0001 False
Newton-CG iter = 1
  Check Convergence
    max |gradient| <= tol: 0.027316554992943484 <= 0.0001 False
Newton-CG iter = 2
  Check Convergence
    max |gradient| <= tol: 0.002026456978832855 <= 0.0001 False
Newton-CG iter = 3
  Check Convergence
    max |gradient| <= tol: 0.0004390195466133051 <= 0.0001 False
Newton-CG iter = 4
  Check Convergence
    max |gradient| <= tol: 2.370092358732281e-06 <= 0.0001 True
  Solver did converge at loss = 0.4634590042946397.
ACCURACY OF THE MODEL: 0.7791411042944786
F1 score: 0.5485025102429454
Precision: 0.7308191782245078
Recall: 0.5576815733108327
Classification Report for Human Annotators
              precision    recall  f1-score   support

         Yes       0.78      0.98      0.87       497
          No       0.68      0.14      0.23       155

    accuracy                           0.78       652
   macro avg       0.73      0.

[Parallel(n_jobs=1)]: Using backend SequentialBackend with 1 concurrent workers.
[Parallel(n_jobs=1)]: Done   1 out of   1 | elapsed:    0.0s finished


In [46]:
print("GPT-5.5")
df34 = pd.read_csv("LLM_annotation2/GPT_5.5_EXIST2023.csv")
log_model(df34["tweet"],df34["GPT_5.5"])

GPT-5.5
Newton-CG iter = 0
  Check Convergence
    max |gradient| <= tol: 0.02338957055214724 <= 0.0001 False
Newton-CG iter = 1
  Check Convergence
    max |gradient| <= tol: 0.0061479788225898505 <= 0.0001 False
Newton-CG iter = 2
  Check Convergence
    max |gradient| <= tol: 0.002908481907250767 <= 0.0001 False
Newton-CG iter = 3
  Check Convergence
    max |gradient| <= tol: 0.0003311789333168315 <= 0.0001 False
Newton-CG iter = 4
  Check Convergence
    max |gradient| <= tol: 1.8867270258227792e-06 <= 0.0001 True
  Solver did converge at loss = 0.5221651314466158.
ACCURACY OF THE MODEL: 0.7730061349693251
F1 score: 0.7661043242195075
Precision: 0.7813348678963362
Recall: 0.7640198205538764
Classification Report for Human Annotators
              precision    recall  f1-score   support

         Yes       0.75      0.87      0.81       353
          No       0.81      0.66      0.73       299

    accuracy                           0.77       652
   macro avg       0.78      0.76 

[Parallel(n_jobs=1)]: Using backend SequentialBackend with 1 concurrent workers.
[Parallel(n_jobs=1)]: Done   1 out of   1 | elapsed:    0.0s finished


In [47]:
print("phi")
df35 = pd.read_csv("LLM_annotation2/phi_EXIST2023.csv")
log_model(df35["tweet"],df35["phi"])

phi
Newton-CG iter = 0
  Check Convergence
    max |gradient| <= tol: 0.10467791411042945 <= 0.0001 False
Newton-CG iter = 1
  Check Convergence
    max |gradient| <= tol: 0.005328071384193943 <= 0.0001 False
Newton-CG iter = 2
  Check Convergence
    max |gradient| <= tol: 0.002246156408286598 <= 0.0001 False
Newton-CG iter = 3
  Check Convergence
    max |gradient| <= tol: 0.0001140680156669141 <= 0.0001 False
Newton-CG iter = 4
  Check Convergence
    max |gradient| <= tol: 2.3825255603266413e-06 <= 0.0001 True
  Solver did converge at loss = 0.5261173859588512.
ACCURACY OF THE MODEL: 0.7361963190184049
F1 score: 0.6825634057971015
Precision: 0.7588045257652802
Recall: 0.6774895430745844
Classification Report for Human Annotators
              precision    recall  f1-score   support

         Yes       0.72      0.93      0.81       401
          No       0.80      0.42      0.55       251

    accuracy                           0.74       652
   macro avg       0.76      0.68      

[Parallel(n_jobs=1)]: Using backend SequentialBackend with 1 concurrent workers.
[Parallel(n_jobs=1)]: Done   1 out of   1 | elapsed:    0.0s finished


In [48]:
print("mistral")
df36 = pd.read_csv("LLM_annotation2/mistral_EXIST2023_Second_time.csv")
log_model(df36["tweet"],df36["mistral"])

mistral
Newton-CG iter = 0
  Check Convergence
    max |gradient| <= tol: 0.13688650306748468 <= 0.0001 False
Newton-CG iter = 1
  Check Convergence
    max |gradient| <= tol: 0.019782699767950697 <= 0.0001 False
Newton-CG iter = 2
  Check Convergence
    max |gradient| <= tol: 0.0031905895498628226 <= 0.0001 False
Newton-CG iter = 3
  Check Convergence
    max |gradient| <= tol: 0.0003966082967085923 <= 0.0001 False
Newton-CG iter = 4
  Check Convergence
    max |gradient| <= tol: 1.8123580533295144e-06 <= 0.0001 True
  Solver did converge at loss = 0.5168769685051414.
ACCURACY OF THE MODEL: 0.7055214723926381
F1 score: 0.6089564174330268
Precision: 0.7724390836591778
Recall: 0.622898542059055
Classification Report for Human Annotators
              precision    recall  f1-score   support

         Yes       0.86      0.27      0.41       249
          No       0.68      0.97      0.80       403

    accuracy                           0.71       652
   macro avg       0.77      0.62  

[Parallel(n_jobs=1)]: Using backend SequentialBackend with 1 concurrent workers.
[Parallel(n_jobs=1)]: Done   1 out of   1 | elapsed:    0.0s finished


In [49]:
print("llama")
df37 = pd.read_csv("LLM_annotation2/llama_EXIST2023_Second_time.csv")
#df2 = df2[df2["llama"].astype(str).str.strip().str.upper().isin(["YES", "NO"])]
log_model(df37["tweet"],df37["llama"])

llama
Newton-CG iter = 0
  Check Convergence
    max |gradient| <= tol: 0.16756134969325154 <= 0.0001 False
Newton-CG iter = 1
  Check Convergence
    max |gradient| <= tol: 0.016085822725486386 <= 0.0001 False
Newton-CG iter = 2
  Check Convergence
    max |gradient| <= tol: 0.0018933596371544558 <= 0.0001 False
Newton-CG iter = 3
  Check Convergence
    max |gradient| <= tol: 0.00033995231928610724 <= 0.0001 False
Newton-CG iter = 4
  Check Convergence
    max |gradient| <= tol: 2.2779295286445584e-06 <= 0.0001 True
  Solver did converge at loss = 0.49998829736728534.
ACCURACY OF THE MODEL: 0.74079754601227
F1 score: 0.6020355462133278
Precision: 0.79453044375645
Recall: 0.6076168929110105
Classification Report for Human Annotators
              precision    recall  f1-score   support

         Yes       0.73      0.98      0.84       442
          No       0.86      0.23      0.37       210

    accuracy                           0.74       652
   macro avg       0.79      0.61     

[Parallel(n_jobs=1)]: Using backend SequentialBackend with 1 concurrent workers.
[Parallel(n_jobs=1)]: Done   1 out of   1 | elapsed:    0.0s finished


**Individual human annotations**

In [50]:
data=pd.read_csv("LLM_annotation2/All_LLM_Prompt2_Combined.csv")

In [51]:
print("anno1")
log_model(data["tweet"],data["anno1"])

anno1
Newton-CG iter = 0
  Check Convergence
    max |gradient| <= tol: 0.11541411042944785 <= 0.0001 False
Newton-CG iter = 1
  Check Convergence
    max |gradient| <= tol: 0.007205968582117316 <= 0.0001 False
Newton-CG iter = 2
  Check Convergence
    max |gradient| <= tol: 0.002172147251901952 <= 0.0001 False
Newton-CG iter = 3
  Check Convergence
    max |gradient| <= tol: 8.320111621902505e-05 <= 0.0001 True
  Solver did converge at loss = 0.5409614679731248.
ACCURACY OF THE MODEL: 0.7055214723926381
F1 score: 0.6089564174330268
Precision: 0.7230024877083701
Recall: 0.6157490722383203
Classification Report for Human Annotators
              precision    recall  f1-score   support

         Yes       0.70      0.94      0.80       415
          No       0.75      0.29      0.41       237

    accuracy                           0.71       652
   macro avg       0.72      0.62      0.61       652
weighted avg       0.72      0.71      0.66       652



[Parallel(n_jobs=1)]: Using backend SequentialBackend with 1 concurrent workers.
[Parallel(n_jobs=1)]: Done   1 out of   1 | elapsed:    0.0s finished


In [52]:
print("anno2")
log_model(data["tweet"],data["anno2"])

anno2
Newton-CG iter = 0
  Check Convergence
    max |gradient| <= tol: 0.06863496932515338 <= 0.0001 False
Newton-CG iter = 1
  Check Convergence
    max |gradient| <= tol: 0.00301226012120653 <= 0.0001 False
Newton-CG iter = 2
  Check Convergence
    max |gradient| <= tol: 0.0025897532605680946 <= 0.0001 False
Newton-CG iter = 3
  Check Convergence
    max |gradient| <= tol: 0.00013488873440077316 <= 0.0001 False
Newton-CG iter = 4
  Check Convergence
    max |gradient| <= tol: 2.868170768247579e-06 <= 0.0001 True
  Solver did converge at loss = 0.549850602197686.
ACCURACY OF THE MODEL: 0.6947852760736196
F1 score: 0.6605721311689841
Precision: 0.7084696586209025
Recall: 0.662399492170969
Classification Report for Human Annotators
              precision    recall  f1-score   support

         Yes       0.68      0.88      0.77       374
          No       0.74      0.44      0.55       278

    accuracy                           0.69       652
   macro avg       0.71      0.66      

[Parallel(n_jobs=1)]: Using backend SequentialBackend with 1 concurrent workers.
[Parallel(n_jobs=1)]: Done   1 out of   1 | elapsed:    0.0s finished


In [53]:
print("anno3")
log_model(data["tweet"],data["anno3"])

anno3
Newton-CG iter = 0
  Check Convergence
    max |gradient| <= tol: 0.11503067484662577 <= 0.0001 False
Newton-CG iter = 1
  Check Convergence
    max |gradient| <= tol: 0.006716472756586325 <= 0.0001 False
Newton-CG iter = 2
  Check Convergence
    max |gradient| <= tol: 0.002263438599859966 <= 0.0001 False
Newton-CG iter = 3
  Check Convergence
    max |gradient| <= tol: 6.894724790150117e-05 <= 0.0001 True
  Solver did converge at loss = 0.5433828813742532.
ACCURACY OF THE MODEL: 0.6763803680981595
F1 score: 0.5901972874832814
Precision: 0.6789373947220663
Recall: 0.6016273530847958
Classification Report for Human Annotators
              precision    recall  f1-score   support

         Yes       0.68      0.92      0.78       403
          No       0.68      0.29      0.40       249

    accuracy                           0.68       652
   macro avg       0.68      0.60      0.59       652
weighted avg       0.68      0.68      0.63       652



[Parallel(n_jobs=1)]: Using backend SequentialBackend with 1 concurrent workers.
[Parallel(n_jobs=1)]: Done   1 out of   1 | elapsed:    0.0s finished


In [54]:
print("anno4")
log_model(data["tweet"],data["anno4"])

anno4
Newton-CG iter = 0
  Check Convergence
    max |gradient| <= tol: 0.06595092024539878 <= 0.0001 False
Newton-CG iter = 1
  Check Convergence
    max |gradient| <= tol: 0.002981153518760735 <= 0.0001 False
Newton-CG iter = 2
  Check Convergence
    max |gradient| <= tol: 0.002261329642325168 <= 0.0001 False
Newton-CG iter = 3
  Check Convergence
    max |gradient| <= tol: 9.54564685939456e-05 <= 0.0001 True
  Solver did converge at loss = 0.5554746062260728.
ACCURACY OF THE MODEL: 0.6671779141104295
F1 score: 0.6265720718224024
Precision: 0.6670088980150581
Recall: 0.6298374613003096
Classification Report for Human Annotators
              precision    recall  f1-score   support

         Yes       0.67      0.86      0.75       380
          No       0.67      0.40      0.50       272

    accuracy                           0.67       652
   macro avg       0.67      0.63      0.63       652
weighted avg       0.67      0.67      0.65       652



[Parallel(n_jobs=1)]: Using backend SequentialBackend with 1 concurrent workers.
[Parallel(n_jobs=1)]: Done   1 out of   1 | elapsed:    0.0s finished


In [55]:
print("anno5")
log_model(data["tweet"],data["anno5"])

anno5
Newton-CG iter = 0
  Check Convergence
    max |gradient| <= tol: 0.08052147239263804 <= 0.0001 False
Newton-CG iter = 1
  Check Convergence
    max |gradient| <= tol: 0.0031455125488034025 <= 0.0001 False
Newton-CG iter = 2
  Check Convergence
    max |gradient| <= tol: 0.002340907999083373 <= 0.0001 False
Newton-CG iter = 3
  Check Convergence
    max |gradient| <= tol: 7.468873311831936e-05 <= 0.0001 True
  Solver did converge at loss = 0.5502774929945315.
ACCURACY OF THE MODEL: 0.6763803680981595
F1 score: 0.6299271270743497
Precision: 0.6934299122628036
Recall: 0.6370259019426456
Classification Report for Human Annotators
              precision    recall  f1-score   support

         Yes       0.66      0.89      0.76       376
          No       0.72      0.38      0.50       276

    accuracy                           0.68       652
   macro avg       0.69      0.64      0.63       652
weighted avg       0.69      0.68      0.65       652



[Parallel(n_jobs=1)]: Using backend SequentialBackend with 1 concurrent workers.
[Parallel(n_jobs=1)]: Done   1 out of   1 | elapsed:    0.0s finished


In [56]:
print("anno6")
log_model(data["tweet"],data["anno6"])

anno6
Newton-CG iter = 0
  Check Convergence
    max |gradient| <= tol: 0.04524539877300614 <= 0.0001 False
Newton-CG iter = 1
  Check Convergence
    max |gradient| <= tol: 0.0029308739003309947 <= 0.0001 False
Newton-CG iter = 2
  Check Convergence
    max |gradient| <= tol: 0.002938861728023572 <= 0.0001 False
Newton-CG iter = 3
  Check Convergence
    max |gradient| <= tol: 0.00023988256425537318 <= 0.0001 False
Newton-CG iter = 4
  Check Convergence
    max |gradient| <= tol: 2.7238819458445286e-06 <= 0.0001 True
  Solver did converge at loss = 0.5577671720496093.
ACCURACY OF THE MODEL: 0.6794478527607362
F1 score: 0.6569414548795992
Precision: 0.6804730473047305
Recall: 0.657095387208289
Classification Report for Human Annotators
              precision    recall  f1-score   support

         Yes       0.68      0.83      0.74       369
          No       0.68      0.49      0.57       283

    accuracy                           0.68       652
   macro avg       0.68      0.66   

[Parallel(n_jobs=1)]: Using backend SequentialBackend with 1 concurrent workers.
[Parallel(n_jobs=1)]: Done   1 out of   1 | elapsed:    0.0s finished
